# ChargeGrid Intelligence — Sprint 4 · avaliação no Colab

Roda o projeto inteiro no Colab, sem instalar nada no computador:

1. clona o repositório e instala as dependências;
2. indexa a base de conhecimento no ChromaDB (`nomic-embed-text`);
3. roda as avaliações — a versão das Sprints 1/2, as 3 iterações do RAG, a comparação de
   modelos e os testes de segurança — com a rubrica de fallback aplicada pelo LLM-juiz;
4. preenche as tabelas dos relatórios e gera `docs/relatorio_evolucao.pdf`;
5. (opcional) sobe a interface web com link público.

**Antes de rodar:** no menu à esquerda, clique no ícone de chave (**Secrets**) e cadastre:

| Nome | Onde conseguir |
|---|---|
| `GROQ_API_KEY` | <https://console.groq.com/keys> |
| `OLLAMA_API_KEY` | <https://ollama.com> → Settings → Keys |
| `GOOGLE_API_KEY` *(recomendado)* | <https://aistudio.google.com/apikey> — grátis, usada se a cota diária da Groq acabar |
| `GITHUB_TOKEN` *(opcional)* | só para enviar os resultados de volta ao GitHub (seção 6) |

Ligue o acesso do notebook a cada segredo. Depois use **Ambiente de execução → Executar tudo**.
A rodada completa leva de 30 a 60 minutos por causa do limite da conta gratuita do Groq.

## 1. Projeto e dependências

In [ ]:
import os

REPO = "https://github.com/pedrosampaiom2007-gif/sprint-4-Jorge.git"
PASTA = "/content/sprint-4-Jorge"

if not os.path.exists(PASTA):
    !git clone -q {REPO} {PASTA}
else:
    !git -C {PASTA} pull -q
%cd {PASTA}
!pip install -q -r requirements.txt

## 2. Chaves

In [ ]:
from google.colab import userdata

CHAVES = ("GROQ_API_KEY", "OLLAMA_API_KEY", "GOOGLE_API_KEY")
for nome in CHAVES:
    try:
        os.environ[nome] = userdata.get(nome) or ""
    except Exception:
        os.environ[nome] = ""
print("chaves carregadas:", [n for n in CHAVES if os.environ.get(n)])

## 3. Conferir as chaves e preparar os embeddings

Testa cada modelo de chat com uma pergunta curta e guarda só os que responderam (a
comparação de modelos da seção 5 usa essa lista). Depois testa o `nomic-embed-text` na
Ollama Cloud; se não funcionar, instala o Ollama dentro do Colab e usa o modelo local.

In [ ]:
from src.llm.provedores import MODELOS_DISPONIVEIS, construir_llm

MODELOS_OK = []
for provedor, modelo in MODELOS_DISPONIVEIS:
    try:
        construir_llm(provedor, modelo, max_tokens=50).invoke("Responda só: ok")
        MODELOS_OK.append(f"{provedor}:{modelo}")
        print(f"ok    {provedor}:{modelo}")
    except Exception as erro:
        print(f"FALHOU {provedor}:{modelo} -> {str(erro)[:150]}")

if not MODELOS_OK:
    raise SystemExit("Nenhum modelo respondeu. Confira as chaves (ou cadastre a GOOGLE_API_KEY) e rode de novo.")

# quem responde e quem julga: Groq se estiver com cota; senão o primeiro que respondeu
GERACAO = "groq:openai/gpt-oss-20b" if "groq:openai/gpt-oss-20b" in MODELOS_OK else MODELOS_OK[0]
JUIZ = "groq:openai/gpt-oss-120b" if "groq:openai/gpt-oss-120b" in MODELOS_OK else GERACAO
print("\nmodelos usados na comparação:", MODELOS_OK)
print("modelo que responde:", GERACAO, "· juiz:", JUIZ)

In [ ]:
import shutil, subprocess, time, urllib.request

def testar_embedding():
    from src.rag.embeddings import criar_embeddings
    return len(criar_embeddings().embed_query("teste de recarga"))

def sh(comando):
    r = subprocess.run(comando, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-1500:], r.stderr[-1500:])
        raise RuntimeError(f"falhou: {comando}")

OLLAMA_LOCAL = "http://127.0.0.1:11434"

def ollama_respondendo():
    try:
        urllib.request.urlopen(OLLAMA_LOCAL, timeout=3)
        return True
    except Exception:
        return False

def garantir_ollama():
    # sobe (ou religa) o servidor local do Ollama, se o notebook estiver usando ele
    if os.environ.get("OLLAMA_EMBED_BASE_URL") != OLLAMA_LOCAL or ollama_respondendo():
        return
    print("Servidor do Ollama fora do ar — religando...")
    binario = shutil.which("ollama") or "/usr/local/bin/ollama"
    subprocess.Popen(f"nohup {binario} serve > /content/ollama.log 2>&1 &", shell=True,
                     env={**os.environ, "OLLAMA_HOST": "127.0.0.1:11434"})
    for _ in range(60):
        if ollama_respondendo():
            return
        time.sleep(1)
    print(open("/content/ollama.log").read()[-2000:])
    raise RuntimeError("O Ollama local não subiu; veja o log acima.")

def subir_ollama_local():
    sh("apt-get -qq update && apt-get -qq install -y zstd pciutils")
    if not shutil.which("ollama"):
        sh("curl -fsSL https://ollama.com/install.sh | sh")
    os.environ["OLLAMA_EMBED_BASE_URL"] = OLLAMA_LOCAL
    garantir_ollama()
    sh(f"{shutil.which('ollama') or '/usr/local/bin/ollama'} pull nomic-embed-text")

try:
    print("Ollama Cloud ok, dimensão:", testar_embedding())
except Exception as erro:
    print("Ollama Cloud não serviu o nomic-embed-text:", str(erro)[:150])
    print("Instalando o Ollama no Colab (1 a 2 minutos)...")
    subir_ollama_local()
    print("Ollama local ok, dimensão:", testar_embedding())

## 4. Testes offline

In [ ]:
!python -m unittest discover -s tests

## 5. Avaliações (um comando só)

Roda a versão das Sprints 1/2, as 3 iterações do RAG, a comparação dos modelos que
responderam na seção 3 e os testes de segurança; aplica a rubrica de fallback
(`evals/fallback/rubrica_manual.md`) com o LLM-juiz da Groq; e gera as tabelas e o PDF.

**Se parar no meio** (limite da Groq, Colab desconectou), rode esta célula de novo: ela
continua de onde parou, sem refazer o que já terminou.

In [ ]:
garantir_ollama()
!python -m evals.rodar_tudo --geracao {GERACAO} --juiz {JUIZ} --modelos {" ".join(MODELOS_OK)}

## 6. Relatórios e PDF

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("evals/historico_scores.md", encoding="utf-8").read()))

In [ ]:
# se a busca falhou com "Failed to connect to Ollama", veja o log do servidor:
# print(open("/content/ollama.log").read()[-3000:])
# baixa resultados, relatórios e o PDF
!zip -qr resultados_sprint4.zip evals/resultados evals/historico_scores.md docs prompts/rag/CHANGELOG.md
from google.colab import files
files.download("resultados_sprint4.zip")
files.download("docs/relatorio_evolucao.pdf")

### Enviar os resultados para o GitHub *(opcional)*

Precisa do segredo `GITHUB_TOKEN` (GitHub → Settings → Developer settings → Personal access
tokens, com permissão de escrita no repositório). Preencha nome e e-mail do autor dos commits.

In [ ]:
NOME_AUTOR = "Pedro Sampaio"
EMAIL_AUTOR = "pedrosampaiom2007@gmail.com"

token = userdata.get("GITHUB_TOKEN")
!git config user.name "{NOME_AUTOR}"
!git config user.email "{EMAIL_AUTOR}"
!git add evals/resultados evals/historico_scores.md docs prompts/rag/CHANGELOG.md
!git commit -q -m "Resultados das avaliações da Sprint 4 (RAGAS, modelos e segurança)"
!git push -q https://{token}@github.com/pedrosampaiom2007-gif/sprint-4-Jorge.git HEAD:main

## 7. Interface web *(opcional)*

In [ ]:
# gera um link público do Gradio (válido enquanto o notebook estiver rodando)
from app.web import construir_interface
construir_interface().launch(share=True)